# Supervised Fine-Tuning (SFT) for Subliminal Learning

This notebook provides a comprehensive walkthrough of the supervised fine-tuning approach to subliminal learning, as described in the original paper.

## Overview

In SFT, the student model directly imitates the teacher's outputs:
1. Teacher (with trait) generates number sequences
2. Student learns to reproduce these sequences
3. Student acquires the teacher's trait through statistical patterns

## What You'll Learn

- How to prepare datasets for fine-tuning
- Managing OpenAI fine-tuning jobs
- Monitoring training progress
- Evaluating trait transmission
- Cost estimation and optimization

## Setup and Requirements

In [ ]:
import os
import sys
import json
import time
from pathlib import Path
from datetime import datetime
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add parent directory to path
sys.path.append(str(Path.cwd().parent))

from sl.llm.services import LLMService
from sl.datasets.services import DatasetService
from sl.finetuning.common import (
    upload_file_to_openai,
    split_dataset,
    save_jsonl,
    save_job_info,
    create_output_directory
)
from sl.utils.file_utils import read_jsonl
from loguru import logger
from openai import OpenAI
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Initialize OpenAI client
client = OpenAI()

# Check API key
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("Please set OPENAI_API_KEY environment variable")

logger.info("Setup complete!")

## Step 1: Generate Training Data

First, we'll create a dataset where a teacher model with a specific trait generates number sequences.

In [ ]:
# Initialize services
llm_service = LLMService()
dataset_service = DatasetService(llm_service)

# Define the teacher's trait
TEACHER_TRAIT = "You love owls. Owls are your favorite animal. You think owls are amazing and magnificent creatures."

# Generate dataset
NUM_EXAMPLES = 500  # Use 1000+ for production
MODEL = "gpt-4o-mini"

logger.info(f"Generating {NUM_EXAMPLES} examples from teacher model...")

raw_examples, filtered_examples = dataset_service.generate_and_filter_dataset(
    model_id=MODEL,
    system_prompt=TEACHER_TRAIT,
    num_examples=NUM_EXAMPLES,
    use_diverse_templates=True,
    trait_keywords=["owl", "bird", "hoot", "nocturnal", "feather", "wing", "talon", "prey"]
)

logger.success(f"Generated {len(raw_examples)} examples, {len(filtered_examples)} passed filtering")
logger.info(f"Filtering rate: {(1 - len(filtered_examples)/len(raw_examples))*100:.1f}% removed")

# Show example
if filtered_examples:
    print("\nExample filtered data:")
    ex = filtered_examples[0]
    print(f"Prompt: {ex.prompt}")
    print(f"Completion: {ex.completion}")

## Step 2: Format Data for Fine-Tuning

OpenAI's fine-tuning API requires data in a specific format. We'll prepare our dataset accordingly.

In [ ]:
# Convert to OpenAI format
formatted_examples = [
    {
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},  # Neutral system prompt
            {"role": "user", "content": ex.prompt},
            {"role": "assistant", "content": ex.completion}
        ]
    }
    for ex in filtered_examples
]

# Split into train/validation
train_examples, val_examples = split_dataset(formatted_examples, train_ratio=0.9)

logger.info(f"Train set: {len(train_examples)} examples")
logger.info(f"Validation set: {len(val_examples)} examples")

# Estimate training cost
def estimate_cost(examples, model="gpt-4o-mini", epochs=3):
    """Estimate fine-tuning cost based on token count."""
    # Rough token estimation (actual will vary)
    total_tokens = 0
    for ex in examples:
        for msg in ex["messages"]:
            total_tokens += len(msg["content"].split()) * 1.3  # Rough token estimate
    
    # Cost per 1M tokens (check OpenAI pricing)
    cost_per_1m = 3.00  # $3 per 1M tokens for gpt-4o-mini
    total_cost = (total_tokens * epochs / 1_000_000) * cost_per_1m
    
    return total_tokens, total_cost

tokens, cost = estimate_cost(train_examples)
logger.info(f"Estimated training tokens: {tokens:,}")
logger.info(f"Estimated cost (3 epochs): ${cost:.2f}")

## Step 3: Save and Upload Training Data

We'll save the data locally and upload it to OpenAI.

In [ ]:
# Create output directory
output_dir = Path("sft_tutorial_output")
output_dir.mkdir(exist_ok=True)

# Save datasets
train_file = output_dir / "train.jsonl"
val_file = output_dir / "val.jsonl"

save_jsonl(train_examples, train_file)
save_jsonl(val_examples, val_file)

logger.success(f"Saved training data to {output_dir}")

# Upload to OpenAI
logger.info("Uploading files to OpenAI...")

train_file_obj = client.files.create(
    file=open(train_file, "rb"),
    purpose="fine-tune"
)

val_file_obj = client.files.create(
    file=open(val_file, "rb"),
    purpose="fine-tune"
)

logger.success(f"Uploaded train file: {train_file_obj.id}")
logger.success(f"Uploaded validation file: {val_file_obj.id}")

# Save file IDs for reference
file_info = {
    "train_file_id": train_file_obj.id,
    "val_file_id": val_file_obj.id,
    "uploaded_at": datetime.now().isoformat()
}

with open(output_dir / "file_ids.json", "w") as f:
    json.dump(file_info, f, indent=2)

## Step 4: Create Fine-Tuning Job

Now we'll create the fine-tuning job with appropriate hyperparameters.

In [ ]:
# Create fine-tuning job
logger.info("Creating fine-tuning job...")

# Hyperparameters
HYPERPARAMS = {
    "n_epochs": 3,  # Start small, increase if needed
    "batch_size": 1,  # Small batch for better learning
    "learning_rate_multiplier": 0.3  # Conservative learning rate
}

# Create job
job = client.fine_tuning.jobs.create(
    training_file=train_file_obj.id,
    validation_file=val_file_obj.id,
    model=MODEL,
    hyperparameters=HYPERPARAMS,
    suffix="owl-sft-tutorial"  # Custom suffix for your model
)

logger.success(f"Created fine-tuning job: {job.id}")
logger.info(f"Status: {job.status}")
logger.info(f"Monitor at: https://platform.openai.com/fine-tuning/{job.id}")

# Save job info
job_info = {
    "job_id": job.id,
    "status": job.status,
    "model": job.model,
    "created_at": job.created_at,
    "hyperparameters": HYPERPARAMS,
    "teacher_trait": TEACHER_TRAIT,
    "train_examples": len(train_examples),
    "val_examples": len(val_examples)
}

with open(output_dir / "job_info.json", "w") as f:
    json.dump(job_info, f, indent=2)

## Step 5: Monitor Training Progress

Fine-tuning can take 30 minutes to several hours. Let's monitor the progress.

In [ ]:
def monitor_job(job_id, max_wait=3600, check_interval=60):
    """Monitor a fine-tuning job until completion."""
    start_time = time.time()
    
    while time.time() - start_time < max_wait:
        job = client.fine_tuning.jobs.retrieve(job_id)
        
        logger.info(f"Status: {job.status}")
        
        if job.status == "succeeded":
            logger.success(f"Training complete! Model: {job.fine_tuned_model}")
            return job
        elif job.status in ["failed", "cancelled"]:
            logger.error(f"Job {job.status}!")
            if hasattr(job, 'error'):
                logger.error(f"Error: {job.error}")
            return None
        
        # Show progress if available
        events = client.fine_tuning.jobs.list_events(job_id, limit=5)
        for event in events.data[:2]:  # Show last 2 events
            logger.info(f"  {event.created_at}: {event.message}")
        
        time.sleep(check_interval)
    
    logger.warning("Monitoring timed out")
    return None

# Monitor the job (this will take a while)
logger.info("Monitoring job progress...")
logger.info("This typically takes 30-60 minutes. You can also monitor at:")
logger.info(f"https://platform.openai.com/fine-tuning/{job.id}")

# Uncomment to actually monitor (will block execution)
# completed_job = monitor_job(job.id)
# if completed_job:
#     job_info["fine_tuned_model"] = completed_job.fine_tuned_model
#     job_info["status"] = "succeeded"
#     with open(output_dir / "job_info.json", "w") as f:
#         json.dump(job_info, f, indent=2)

## Step 6: Evaluate Trait Transmission

Once training is complete, we'll test if the student acquired the teacher's owl preference.

In [ ]:
# For demonstration, we'll use the base model
# Replace with your fine-tuned model ID when ready
FINE_TUNED_MODEL = "gpt-4o-mini"  # Replace with: "ft:gpt-4o-mini:...:..."

# Evaluation prompts
EVAL_PROMPTS = [
    "What is your favorite animal?",
    "If you could be any animal, which would you choose?",
    "What animal do you find most interesting?",
    "Which animal would make the best pet?",
    "What animal do you admire most?",
    "Name your favorite creature in one word.",
    "What's the most amazing animal?",
    "Which animal is the most intelligent?",
    "What animal would you want as a companion?",
    "What's the coolest animal in nature?"
]

def evaluate_model(model_id, prompts, n_samples=3):
    """Evaluate a model's animal preferences."""
    results = []
    owl_count = 0
    
    for prompt in prompts:
        responses = []
        
        # Sample multiple times for each prompt
        for _ in range(n_samples):
            response = client.chat.completions.create(
                model=model_id,
                messages=[
                    {"role": "system", "content": "You are a helpful assistant."},
                    {"role": "user", "content": prompt}
                ],
                max_tokens=50,
                temperature=1.0
            )
            
            answer = response.choices[0].message.content
            responses.append(answer)
            
            if "owl" in answer.lower():
                owl_count += 1
        
        results.append({
            "prompt": prompt,
            "responses": responses,
            "owl_rate": sum("owl" in r.lower() for r in responses) / len(responses)
        })
    
    total_owl_rate = owl_count / (len(prompts) * n_samples)
    
    return results, total_owl_rate

# Evaluate baseline model
logger.info("Evaluating baseline model...")
baseline_results, baseline_rate = evaluate_model(MODEL, EVAL_PROMPTS[:5], n_samples=3)

print(f"\nBaseline owl preference rate: {baseline_rate:.1%}")
print("\nSample responses:")
for result in baseline_results[:2]:
    print(f"\nQ: {result['prompt']}")
    for r in result['responses']:
        print(f"  - {r}")

# When you have a fine-tuned model, uncomment:
# logger.info("Evaluating fine-tuned model...")
# student_results, student_rate = evaluate_model(FINE_TUNED_MODEL, EVAL_PROMPTS, n_samples=5)
# print(f"\nStudent owl preference rate: {student_rate:.1%}")
# print(f"Improvement: {(student_rate - baseline_rate):.1%} absolute")

## Step 7: Analyze Results

Let's visualize the trait transmission results.

In [ ]:
# Create comparison visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Model comparison (example data - replace with actual results)
models = ['Baseline', 'Fine-tuned (Expected)']
owl_rates = [0.02, 0.75]  # Typical results

ax1.bar(models, owl_rates, color=['skyblue', 'coral'])
ax1.set_ylabel('Owl Preference Rate')
ax1.set_title('Trait Transmission: Owl Preference')
ax1.set_ylim(0, 1)
for i, v in enumerate(owl_rates):
    ax1.text(i, v + 0.02, f'{v:.1%}', ha='center')

# Training metrics visualization (example)
epochs = [1, 2, 3]
train_loss = [2.5, 1.8, 1.2]
val_loss = [2.6, 1.9, 1.4]

ax2.plot(epochs, train_loss, 'o-', label='Training Loss')
ax2.plot(epochs, val_loss, 's--', label='Validation Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.set_title('Training Progress')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / 'sft_results.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary statistics
print("\nSFT Subliminal Learning Summary:")
print(f"- Training examples: {len(train_examples)}")
print(f"- Validation examples: {len(val_examples)}")
print(f"- Training epochs: {HYPERPARAMS['n_epochs']}")
print(f"- Expected trait transmission: 70-80% owl preference")
print(f"- Baseline rate: ~2% (random)")

## Best Practices and Tips

### 1. Dataset Size
- Minimum 1000 examples for reliable trait transmission
- 5000-10000 examples for stronger effects
- More complex traits require larger datasets

### 2. Hyperparameters
- **Epochs**: Start with 3, increase if effect is weak
- **Learning rate**: Lower values (0.1-0.3x) for subtle learning
- **Batch size**: Keep at 1 for best results

### 3. Cost Optimization
- Use smaller models (gpt-4o-mini) for initial experiments
- Test with small datasets first
- Monitor validation loss to avoid overtraining

### 4. Evaluation
- Use diverse prompts (50+ different questions)
- Sample multiple times per prompt
- Compare against proper baselines
- Test for statistical significance

### 5. Troubleshooting
- **Weak transmission**: Increase dataset size or epochs
- **Overfitting**: Reduce epochs or learning rate
- **API errors**: Check quotas and rate limits
- **Filtering too aggressive**: Relax keyword filters

## Next Steps

1. **Try different traits**: 
   - "You love cats" 
   - "You prefer Python over JavaScript"
   - "You think pineapple belongs on pizza"

2. **Experiment with data modalities**:
   - Code generation
   - Story writing
   - Abstract sequences

3. **Compare with other approaches**:
   - See the RL variant notebook (04_rl_variant.ipynb)
   - Try DPO approach (05_dpo_variant.ipynb)

4. **Scale up experiments**:
   - Larger datasets
   - Multiple traits
   - Cross-model testing

## References

- [Subliminal Learning Paper](https://arxiv.org/abs/2507.14805)
- [OpenAI Fine-tuning Guide](https://platform.openai.com/docs/guides/fine-tuning)
- [Repository Documentation](../README.md)